<a href="https://colab.research.google.com/github/kenzoyanome/analisis_desconnecta/blob/main/Analisis_Desconnecta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis Desconnecta
Evaluar el **comportamiento de los clientes** de una empresa de telecomunicaciones en Latinoamerica, *Desconnecta*.  
Trabajaremos con informacion registrada **hasta el año 2024**, lo cual permitira analizar el comportamiento del negocio dentro de ese periodo.

Trabajaremos con tres datasets:  
- **plans.csv** → informacion de los planes actuales (precio, minutos incluidos, GB incluidos, costo por extra)
- **users.csv** → informacion de los clientes (edad, ciudad, fecha de registro, plan, churn)
- **usage.csv** → detalle del **uso real** de los servicios (llamadas y mensajes)

**Exploraremos**, **limpiaremos** y **analizaremos** estos datos para construir un **perfil estadistico** de los clientes, detectar **comportamientos atipicos** y crear **segmentos de clientes**.

Este analisis permitira **identificar patrones de consumo**, **diseñar estrategias de retencion** y **sugerir mejoras en los planes** ofrecidos por la empresa.  
<br>
<details>
<summary>Haz clic para ver cuantos puntos principales son:</summary>
Trabajaremos en 8 puntos distintos

---
## 1.0 Cargar y explorar
Antes de limpiar o combinar los datos, es necesario **familiarizarse con la estructura de los tres datasets**.  
En esta etapa, validaremos que los archivos se carguen correctamente, conoceremos sus columnas y tipos de datos, y detectaremos posibles inconsistencias.

---
### 1.1 Carga de datos y vista rapida
**Objetivo:**  
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

**Instrucciones:**  
- Importa las librerias necesarias
- Carga los archivos CSV:
  - `/plans.csv`
  - `/users_latam.csv`
  - `/usage.csv`
- Guarda los DataFrames en variables: `plans`, `users`, `usage`.  
- Muestra las primeras filas de cada DataFrame

In [14]:
# importar librerías
import pandas            as pd
import numpy             as np
import seaborn           as sns
import matplotlib.pyplot as plt

# Otras configuraciones
import warnings                                                  #MANEJO DE WARNINGS - ADVERTENCIAS
warnings.simplefilter(action='ignore', category = FutureWarning) #QUITAR WARNINGS MOLESTOS
pd.set_option('display.max_columns', None)                       #ELIMINA LIMITES DE PANDAS PARA MOSTRAR COLUMNAS
pd.set_option('display.max_rows', None)                          #ELIMINA LIMITES DE PANDAS PARA MOSTRAR FILAS
pd.set_option('display.max_colwidth', None)                      #AUTOAJUSTA ANCHO DE COLUMNAS
pd.set_option('display.float_format', lambda x: '%.3f' % x)      #PERMITE EVADIR EL MOSTRAR NÚMEROS CON NOTACIÓN CINETÍFICA
plt.rcParams['figure.dpi'] = 140                                 #NIVEL DE RESOLUCIÓN.

In [17]:
# cargar archivos
plans = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/analisis_desconnecta/refs/heads/main/plans.csv')
users = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/analisis_desconnecta/refs/heads/main/users_latam.csv')
usage = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/analisis_desconnecta/refs/heads/main/usage.csv')

In [19]:
# mostrar las primeras filas de cada dataset
display(plans.head())
display(users.head())
display(usage.head())

,plan_name,messages_included,gb_per_month,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute
0,Basico,100,5,100,12,1.200,0.080,0.100
1,Premium,500,20,600,25,1.000,0.050,0.070


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
0,10000,Carlos,Garcia,38,Medellín,2022-01-01 00:00:00.000000000,Basico,NaN
1,10001,Mateo,Torres,53,?,2022-01-01 06:34:17.914478619,Basico,NaN
2,10002,Sofia,Ramirez,57,CDMX,2022-01-01 13:08:35.828957239,Basico,NaN
3,10003,Mateo,Ramirez,69,Bogotá,2022-01-01 19:42:53.743435858,Premium,NaN
4,10004,Mateo,Torres,63,GDL,2022-01-02 02:17:11.657914478,Basico,NaN


,id,user_id,type,date,duration,length
0,1,10332,call,2024-01-01 00:00:00.000000000,0.090,NaN
1,2,11458,text,2024-01-01 00:06:30.969774244,NaN,39.000
2,3,11777,text,2024-01-01 00:13:01.939548488,NaN,36.000
3,4,10682,call,2024-01-01 00:19:32.909322733,1.530,NaN
4,5,12742,call,2024-01-01 00:26:03.879096977,4.840,NaN


---
### 1.2 Exploracion de la estructura de los datasets
**Objetivo:**  
Conocer la estructura de cada dataset, revisar cuantas filas y columnas tienen, identificar los tipos de datos de cada columna y detectar posibles inconsistencias o valores nulos antes de iniciar el analisis.

**Instrucciones:**  
Revisa el numero de filas y columnas de cada dataset.  
Obtener un resumen completo de columnas, tipos de datos y valores no nulos.

In [28]:
# Conteo de filas y columnas
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

plans (2, 8)
users (4000, 8)
usage (40000, 6)


In [29]:
# Resumen completo de columnas, tipos de datos y valores no nulos
[plans.info(), users.info(), usage.info()]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   plan_name          2 non-null      object 
 1   messages_included  2 non-null      int64  
 2   gb_per_month       2 non-null      int64  
 3   minutes_included   2 non-null      int64  
 4   usd_monthly_pay    2 non-null      int64  
 5   usd_per_gb         2 non-null      float64
 6   usd_per_message    2 non-null      float64
 7   usd_per_minute     2 non-null      float64
dtypes: float64(3), int64(4), object(1)
memory usage: 260.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     4000 non-null   int64 
 1   first_name  4000 non-null   object
 2   last_name   4000 non-null   object
 3   age         4000 non-null   int64 
 4   ci

[None, None, None]

In [31]:
# Otra forma de mostrarlos seria:
import io
from IPython.display import display, HTML

def capture_info(df):
    buffer = io.StringIO()
    df.info(buf=buffer)
    return buffer.getvalue().replace("\n", "<br>")

html = f"""
<div style="display: flex; gap: 20px;">
  <div>{capture_info(plans)}</div>
  <div>{capture_info(users)}</div>
  <div>{capture_info(usage)}</div>
</div>
"""

display(HTML(html))

---
## 2.0 Identificación de problemas de calidad de datos
### 2.1 Revisión de valores nulos
**Objetivo:**  
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

**Instrucciones:**  
Cuenta valores nulos por columna para cada dataset.
Calcula la proporción de nulos por columna para cada dataset.
El dataset plans solamente tiene 2 renglones y se puede observar que no tiene ausentes, por ello no necesita exploración adicional.

In [32]:
# cantidad de nulos para users
print(users.isna().sum()) # Cantidad de valores nulos
print(users.isna().mean()) # Proporción de valores nulos

user_id          0
first_name       0
last_name        0
age              0
city           469
reg_date         0
plan             0
churn_date    3534
dtype: int64
user_id      0.000
first_name   0.000
last_name    0.000
age          0.000
city         0.117
reg_date     0.000
plan         0.000
churn_date   0.883
dtype: float64


In [33]:
# cantidad de nulos para usage
print(usage.isna().sum()) # Cantidad de valores nulos
print(usage.isna().mean()) # Proporción de valores nulos

id              0
user_id         0
type            0
date           50
duration    22076
length      17896
dtype: int64
id         0.000
user_id    0.000
type       0.000
date       0.001
duration   0.552
length     0.447
dtype: float64


**Comentarios y Notas:**  
¿Qué columnas tienen valores faltantes y en qué proporción?
- users: 'city' muestra un 11.7% de datos faltantes y 'churn_date' muestra un 88.4%
- usage: 'date' solo muestra 50 datos faltantes de los 40,000, 'duration' tiene 55.2% y 'length' un 44.7% the missings

Imputar, eliminar, ignorar?
- users - city: podriamos dejarlos como nulos a menos que tengamos que segmentar por 'city', entonces se recomendaria imputar datos o eliminar
- users - churn_date: debido a que esa columna nos ayuda a calcular la retencion, pudieramos dejarlos como nulos
- usage - date: ya que estamos trabajando solo en el periodo del 2024 y son pocos missings, pudieramos imputarlo
- usage - duration: podriamos dejarlos como nulos ya que estos solo muestran la duracion de llamadas en caso de existir
- usage - length: podriamos dejarlos como nulos ya que estos solo muestran la longituds de mensajes en caso de existir


---
### 2.2 Detección de valores invalidos y sentinels
**Objetivo:**  
Identificar sentinels: valores que no deberían estar en el dataset.

**Instrucciones:**
- Explora las columnas numericas con **un resumen estadistico**
- Explora las columnas categoricas **relevantes**, revisando sus valores unicos

El dataset `plans` solamente tiene 2 renglones, por ello no necesita exploración adicional.

In [37]:
# explorar columnas numéricas de users
users.describe()

,user_id,age
count,4000.000,4000.000
mean,11999.500,33.740
std,1154.845,123.232
min,10000.000,-999.000
25%,10999.750,32.000
50%,11999.500,47.000
75%,12999.250,63.000
max,13999.000,79.000


**Comentarios y Notas:**  
- user_id: no existe discrepancia, los datos estan dentro de un mismo rango claro
- age: muestra minimos imposibles

In [38]:
# explorar columnas numéricas de usage
usage.describe()

,id,user_id,duration,length
count,40000.000,40000.000,17924.000,22104.000
mean,20000.500,12002.406,5.202,52.127
std,11547.150,1157.280,6.843,56.611
min,1.000,10000.000,0.000,0.000
25%,10000.750,10996.000,1.438,37.000
50%,20000.500,12013.000,3.500,50.000
75%,30000.250,13005.000,6.990,64.000
max,40000.000,13999.000,120.000,1490.000


**Comentarios y Notas:**  
- id y user_id muestran valores correctos
- duration y legth muestran minimos en 0.0 (nulos) y valores maximos muy elevados

In [53]:
# explorar columnas categóricas de users
columnas_user = ['city', 'plan']
display(users[columnas_user].value_counts())
# explorar columna categórica de usage
usage['type'].value_counts()

,,count
city,plan,
Bogotá,Basico,522
CDMX,Basico,474
Medellín,Basico,398
GDL,Basico,298
Bogotá,Premium,286
MTY,Basico,275
Cali,Basico,262
CDMX,Premium,256
Medellín,Premium,218


,count
type,
text,22092
call,17908


**Comentarios y Notas:**  
En que columnas encontraste valores invalidos o sentinels?
- city: muestra valores invalidos dentro de ambos planes

Que accion tomarias?
- Pudieramos imputar con la ciudad mas frecuente de cada plan

---
### 2.3 Revision y estandarizacion de fechas
**Objetivo:**  
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

**Instrucciones:**  
- Convierte las columnas de fecha a tipo fecha y asegurate de que el codigo sea a prueba de errores.
- Revisa cuantas veces aparece cada año.
- Identifica fechas imposibles (ej. años futuros o negativos).

Toma en cuenta que tenemos datos registrados hasta el año 2024.

In [56]:
# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'], errors='coerce')
# Convertir a fecha la columna `date` de usage
usage['date'] = pd.to_datetime(usage['date'], errors='coerce')
# Revisar los años presentes en `reg_date` de users
print(users['reg_date'].dt.year.value_counts())
# Revisar los años presentes en `date` de usage
usage['date'].dt.year.value_counts()

reg_date
2024    1330
2023    1316
2022    1314
2026      40
Name: count, dtype: int64


,count
date,
2024.000,39950


**Comentarios y Notas:**  
reg_date: podemos observar que existen registros de usuarios en años distintos. Los años 2025 en adelante los sustituiremos por Na.   
date: las fechas aparecen del 2024 solamente, que es sobre las que trabajaremos.

---
## 3.0 Limpieza basica de datos
### 3.1 Corregir sentinels y fechas imposibles
**Objetivo:**  
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

**Instrucciones:**  
Reemplaza sentinels (-999, ?).  
Marca como nulas las fechas fuera de rango.


In [65]:
# Reemplazar -999 por la mediana de age
age_mediana = users['age'].median()
users['age'] = users['age'].replace((-999),age_mediana)
# Verificar cambios
users['age'].describe()

,age
count,4000.000
mean,48.122
std,17.690
min,18.000
25%,33.000
50%,47.000
75%,63.000
max,79.000


In [88]:
# Reemplazar ? por NA en city
users['city'] = users['city'].replace(("?"),pd.NA)
# Verificar cambios
print(users['city'].value_counts())
users['city'].isna().sum()

city
Bogotá      808
CDMX        730
Medellín    616
GDL         450
Cali        424
MTY         407
Name: count, dtype: int64


np.int64(565)

In [89]:
# Marcar fechas futuras como NA para reg_date
users.loc[users['reg_date'].dt.year > 2024, 'reg_date'] = pd.NA

# Verificar cambios
print(users['reg_date'].dt.year.value_counts())
users['reg_date'].isna().sum()

reg_date
2024.000    1330
2023.000    1316
2022.000    1314
Name: count, dtype: int64


np.int64(40)

---
### 3.2 Corregir sentinels y fechas imposibles
**Objetivo:**  
Decidir qué hacer con los valores nulos segun su proporcion y relevancia.

**Instrucciones:**  
Verifica los nulos en duration y length son **MCAR / MAR / MNAR**, revisando si dependen de la columna type.
- Si confirmas que son MAR, **déjalos como nulos** y justifica la decisión.

In [90]:
# Verificación MAR en usage (Missing At Random) para duration
usage.groupby('type')['duration'].agg(lambda x: x.isna().mean()).sort_values(ascending=False).head()

,duration
type,
text,0.999
call,0.000


In [91]:
# Verificación MAR en usage (Missing At Random) para length
usage.groupby('type')['length'].agg(lambda x: x.isna().mean()).sort_values(ascending=False).head()

,length
type,
call,0.999
text,0.000


**Comentarios y Notas:**  
duration: Los nulos en duration dependen completamente de la variable type, por lo tanto NO son MAR  
legth: Los nulos en length también dependen completamente de type, por lo tanto: NO son MAR  
Los valores faltantes son estructurales (missing by design), ya que dependen directamente del tipo de interaccion. No corresponden a mecanismos MCAR, MAR o MNAR  
NO son errores, no deben imputarse ni eliminarse, son parte natural del diseño del dataset

duration falta cuando type = text  
length falta cuando type = call

---
## 4.0 Summary statistics de uso por usuario
### 4.1 Agrupacion por comportamiento de uso
**Objetivo:**  
Resumir las variables clave de la tabla `usage` **por usuario**, creando metricas que representen su comportamiento real de uso historico.

**Instrucciones:**
1. Construir una tabla agregada de `usage` por user_id que incluya:
- numero total de mensajes
- numero total de llamadas
- total de minutos de llamadas
2. Renombra las columnas para que tengan nombres claros:
- `cant_mensajes`
- `cant_llamadas`
- `cant_minutos_llamada`
3. Combina esta tabla con `users`.

In [94]:
# Columnas auxiliares
usage["is_text"] = (usage["type"] == "text").astype(int) #conocer el total de mensajes
usage["is_call"] = (usage["type"] == "call").astype(int) #conocer el total de llamadas
# Agrupar información por usuario
usage_agg = usage.groupby('user_id')[['is_text', 'is_call', 'duration']].sum().reset_index()
# Otra opcion de hacerlo seria:
#    usage_agg = usage.groupby('user_id').agg({
#        'is_text' : 'sum',                                   # Total de mensajes
#        'is_call' : 'sum',                                   # Total de llamadas
#        'duration': 'sum'                                    # Total de minutos de llamadas
#    }).reset_index()

# observar resultado
usage_agg.head()

,user_id,is_text,is_call,duration
0,10000,7,3,23.700
1,10001,5,10,33.180
2,10002,5,2,10.740
3,10003,11,3,8.990
4,10004,4,3,8.010


In [95]:
# Renombrar columnas
usage_agg = usage_agg.rename(columns={'is_text':'cant_mensajes', 'is_call':'cant_llamadas', 'duration':'cant_minutos_llamada'})
# observar resultado
usage_agg.head(3)

,user_id,cant_mensajes,cant_llamadas,cant_minutos_llamada
0,10000,7,3,23.700
1,10001,5,10,33.180
2,10002,5,2,10.740


In [109]:
# Combinar la tabla agregada con el dataset de usuarios
user_profile = pd.merge(users, usage_agg, on=['user_id'], how='left')
user_profile.head()

,user_id,first_name,last_name,age,city,reg_date,plan,churn_date,cant_mensajes,cant_llamadas,cant_minutos_llamada
0,10000,Carlos,Garcia,38,Medellín,2022-01-01 00:00:00.000000000,Basico,NaN,7.000,3.000,23.700
1,10001,Mateo,Torres,53,<NA>,2022-01-01 06:34:17.914478619,Basico,NaN,5.000,10.000,33.180
2,10002,Sofia,Ramirez,57,CDMX,2022-01-01 13:08:35.828957239,Basico,NaN,5.000,2.000,10.740
3,10003,Mateo,Ramirez,69,Bogotá,2022-01-01 19:42:53.743435858,Premium,NaN,11.000,3.000,8.990
4,10004,Mateo,Torres,63,GDL,2022-01-02 02:17:11.657914478,Basico,NaN,4.000,3.000,8.010


In [111]:
# Verificar si hay usuarios sin datos de uso
user_profile[user_profile[['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']].isna().any(axis=1)]

,user_id,first_name,last_name,age,city,reg_date,plan,churn_date,cant_mensajes,cant_llamadas,cant_minutos_llamada
1082,11082,Luis,Gomez,39,CDMX,2022-10-24 06:31:03.465866468,Basico,NaN,NaN,NaN,NaN


### 4.2 4.2 Resumen estadistico por usuario durante el 2024

**Objetivo:**  
Analizar las columnas numericas y categoricas de los usuarios, para identificar rangos, valores extremos y distribucion de los datos antes de continuar con el analisis.

**Instrucciones:**  
1. Para las columnas **numericas** relevantes, obten un resumen estadistico (media, mediana, minimo, maximo, etc.).  
2. Para la columna **categorica** plan, revisa la distribucion en **porcentajes** de cada categoria.